In [79]:
import json
import os
import numpy as np
import random
import string
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from tokenizers import Tokenizer
from comment_classifier.model import commentClassifier
from comment_generator.model_v4 import Generator

In [22]:
def coin_preprocess(tokenizer, comment):
    def count_punc_num(comment, comment_len):
        count = lambda l1, l2: sum([1 for x in l1 if x in l2])
        punc_num = count(comment, set(string.punctuation))
        digits_num = count(comment, set(string.digits))
        return (punc_num + digits_num) / comment_len

    comment_tokens = tokenizer.tokenize(comment)
    cc_tokens = [tokenizer.cls_token] + comment_tokens + [tokenizer.sep_token]
    cc_ids = tokenizer.convert_tokens_to_ids(cc_tokens)
    cc_att_mask = [1] * len(cc_tokens)
    punc_num = count_punc_num(comment, len(comment.strip().split()))
    if len(comment.strip().split()) < 3:
        comment_len = 1
    else:
        comment_len = 0    
    return torch.tensor(cc_ids).unsqueeze(0), torch.tensor(cc_att_mask).unsqueeze(0), \
           torch.tensor(comment_len).unsqueeze(0), torch.tensor(punc_num).unsqueeze(0)

In [80]:
def dome_preprocess(tokenizer, code, exemplar, intent):
    intent2id = {'what': 0, 'why': 1, 'usage': 2, 'done': 3, 'property': 4}
    intent2bos_id = {'what': "[WHAT/]", 'why': "[WHY/]", 'usage': "[USAGE/]", 'done': "[DONE/]",
                          'property': "[PROP/]"}
    intent2cls_id = {'what': "[/WHAT]", 'why': "[/WHY]", 'usage': "[/USAGE]", 'done': "[/DONE]",
                          'property': "[/PROP]"}
    code_token = tokenizer.encode(code.strip()).ids + [tokenizer.token_to_id(intent2cls_id[intent])]
    code_len = len(code_token)
    exemplar_token = tokenizer.encode(exemplar.strip()).ids
    exemplar_len = len(exemplar_token)
    intent_id = intent2id[intent]
    bos_id = [tokenizer.token_to_id(intent2bos_id[intent])]
    
    return torch.tensor(code_token).unsqueeze(0), torch.tensor(exemplar_token).unsqueeze(0), \
           torch.tensor(bos_id).unsqueeze(0), torch.tensor(code_len).unsqueeze(0), \
           torch.tensor(exemplar_len).unsqueeze(0), torch.tensor(intent_id).unsqueeze(0)

## Comment-Intent Labeling Tool (COIN)

In [5]:
# using COIN to predict intent for a given comment
# 1. load the well-trained COIN
classifier = commentClassifier('./comment_classifier/pretrained_codebert', 6, 0.2)
classifier.load_state_dict(torch.load("./comment_classifier//saved_model/comment_classifier.pkl"))
classifier.cuda()

commentClassifier(
  (codeBert): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0): RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNor

In [36]:
# 2. input a comment and preprocess it
comment1 = 'Starts the background initialization'
comment2 = 'After the construction of a BackgroundInitializer() object it start() method has to be called .'
tokenizer = AutoTokenizer.from_pretrained('./comment_classifier/pretrained_codebert')
cc_ids1, cc_att_mask1, comment_len1, punc_num1 = coin_preprocess(tokenizer, comment1)
cc_ids2, cc_att_mask2, comment_len2, punc_num2 = coin_preprocess(tokenizer, comment2)

In [37]:
# 3. predict the intent for the comment
class_name = ['what', 'why', 'how-to-use', 'how-it-is-done', 'property', 'others']
logits1 = classifier(cc_ids1.cuda(), cc_att_mask1.cuda(), comment_len1.cuda(), punc_num1.cuda())
prediction1 = class_name[int(torch.argmax(logits1, 1))]
prediction1 #'Starts the background initialization'

'what'

In [66]:
classifier.eval()
logits2 = classifier(cc_ids2.cuda(), cc_att_mask2.cuda(), comment_len2.cuda(), punc_num2.cuda())
prediction2 = class_name[int(torch.argmax(logits2, 1))]
prediction2 # 'After the construction of a BackgroundInitializer() object it start() method has to be called .'

NameError: name 'classifier' is not defined

## Comment Comment Generator (DOME)

In [70]:
# using DOME to generate a comment given a code snippet and the user intent
# 1. load the well-trained DOME
class Config(object):
    def __init__(self):
        self.cuda = True
        self.dataset = 'funcom'
        self.bpe_model = f'./comment_generator/dataset/{self.dataset}/bpe_tokenizer_all_token.json'
        self.tokenizer = Tokenizer.from_file(self.bpe_model)
        self.vocab_size = self.tokenizer.get_vocab_size()
        self.eos_token = self.tokenizer.token_to_id('[EOS]')

        self.d_model = 512
        self.d_intent = 256
        self.d_ff = 2048
        self.head_num = 8
        self.layer_num = 4
        self.max_code_num = 100
        self.max_token_num = 20
        self.max_stat_num = 10
        self.max_comment_len = 15
        self.clip_dist_code = 8
        self.clip_dist_stat = 8
        self.intent_num = 5
        self.beam_width = 4
        self.lr = 1e-4
        self.batch_size = 128
        self.dropout = 0.2
        self.epochs = 20
config = Config()
generator = Generator(config.d_model, config.d_intent, config.d_ff, config.head_num, config.layer_num, config.vocab_size,
                      config.max_stat_num, config.max_comment_len, config.clip_dist_code, config.clip_dist_stat,
                      config.eos_token, config.intent_num, config.dropout, None)
generator.load_state_dict(torch.load("./comment_generator/saved_model/funcom/dome_parameters.pkl"))
generator.cuda()

30
25
20
15


Generator(
  (share_embedding): Embedding(50014, 512)
  (intent_embedding): Embedding(5, 256)
  (comment_pos_embedding): Embedding(17, 512)
  (code_encoder): EncoderWithRPR(
    (layers): ModuleList(
      (0): EncoderBlockWithRPR(
        (self_attention): MultiHeadAttentionWithRPR(
          (attention_rpr): DotProductAttentionWithRPR(
            (dropout): Dropout(p=0.2, inplace=False)
          )
          (W_q): Linear(in_features=512, out_features=512, bias=False)
          (W_k): Linear(in_features=512, out_features=512, bias=False)
          (W_v): Linear(in_features=512, out_features=512, bias=False)
          (W_o): Linear(in_features=512, out_features=512, bias=False)
          (relative_pos_v): Embedding(17, 64)
          (relative_pos_k): Embedding(17, 64)
        )
        (feedForward): PositionWiseFFN(
          (W_1): Linear(in_features=512, out_features=2048, bias=True)
          (W_2): Linear(in_features=2048, out_features=512, bias=True)
          (dropout): Dropou

In [89]:
# 2. data preprocessing
code = "public int start ( ) throws IO Exception { synchronized ( this . monitor ) { Assert . state ( this . server Thread == null , "Server already started" ) ; Server Socket Channel server Socket Channel = Server Socket Channel . open ( ) ; server Socket Channel . socket ( ) . bind ( new Inet Socket Address ( this . listen Port ) ) ; int port = server Socket Channel . socket ( ) . get Local Port ( ) ; logger . trace ( Log Message . format ( "Listening for TCP traffic to tunnel on port %s" , port ) ) ; this . server Thread = new Server Thread ( server Socket Channel ) ; this . server Thread . start ( ) ; return port ; } }''
exemplar = "main processing method for the timeserver object"
intent = 'why'
code, exemplar, bos, code_valid_len, exemplar_valid_len, intent = dome_preprocess(config.tokenizer, code, exemplar, intent)
code, exemplar, bos, code_valid_len, exemplar_valid_len, intent

SyntaxError: invalid syntax (3259642385.py, line 2)

In [88]:
# 3. generate the intent-aware comment
generator.eval()
comment_what = generator(code.cuda(), exemplar.cuda(), bos.cuda(), code_valid_len.cuda(), exemplar_valid_len.cuda(), intent.cuda())[0]
config.tokenizer.decode(comment_what)

'gets the urls from the specified thread'